# 0. Install and Import Dependencies

In [1]:
!pip install mediapipe opencv-python


[notice] A new release of pip is available: 23.2.1 -> 24.1.2
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [1]:
import cv2
import mediapipe as mp
import numpy as np
import requests 
from collections import deque
import threading
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# 1. Determining Joints

<img src="https://i.imgur.com/3j8BPdc.png" style="height:300px" >

# 2. Calculate Angles

In [2]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

# 3. Note Determination

In [4]:
 
# Define a lookup table for note determination
NOTE_LOOKUP_TABLE = {
    # angle elbow, angle shoulder
    ((135, 180), (135, 180)): "0", # Start and end pose
    ((135, 180), (  0,  45)): "1", # Ignored default pose
    ((135, 180), ( 45,  90)): "2",
    (( 90, 135), (  0, 180)): "3",
    (( 45,  90), (  0, 180)): "4",
    ((  0,  45), (  0, 180)): "5",
}

# Determine the note based on the angles using a lookup table
def determine_note(angle_elbow, angle_shoulder):
    for ((elbow_min, elbow_max), (shoulder_min, shoulder_max)), note in NOTE_LOOKUP_TABLE.items():
        if elbow_min <= angle_elbow < elbow_max and shoulder_min <= angle_shoulder < shoulder_max:
            return note
    return "404"

# 4. Combine Notes (to send to server)

In [5]:
def combination_result(manuale, tempo):
    manuale = int(manuale)
    tempo = int(tempo)

    if manuale == 0 and tempo == 0:
        return 0
    
    if 1 <= manuale <= 5 and 1 <= tempo <= 5:
        return (manuale - 1) * 5 + tempo
    
    return None


# 5. Main

In [40]:
# Main function to process video feed
def process_video():
    cap = cv2.VideoCapture(0)
    url = 'http://10.10.35.129:8080/producer'
    command_history = deque(maxlen=25)
    command_allowed = False
    last_command = 0
    
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Recolor image to RGB
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            # Make detection
            results = pose.process(image)

            # Recolor back to BGR
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            # Extract landmarks
            try:
                landmarks = results.pose_landmarks.landmark

                # Get coordinates
                hipLeft       = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
                hipRight      = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
                shoulderLeft  = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
                shoulderRight = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
                elbowLeft     = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
                elbowRight    = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
                wristLeft     = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
                wristRight    = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

                # Calculate angle
                angleElbowLeft     = calculate_angle(shoulderLeft, elbowLeft, wristLeft)
                angleElbowRight    = calculate_angle(shoulderRight, elbowRight, wristRight)
                angleShoulderLeft  = calculate_angle(hipLeft, shoulderLeft, elbowLeft)
                angleShoulderRight = calculate_angle(hipRight, shoulderRight, elbowRight)

                # Determine notes
                noteManuale = determine_note(angleElbowRight, angleShoulderRight)
                noteTempo   = determine_note(angleElbowLeft , angleShoulderLeft )
                command = combination_result(noteManuale, noteTempo )
                if (command==0 or command==2 or command==3 or command==5 or command==6 or command==11 or command==16 or command==21):
                    command_history.append(command)
                    if len(command_history) == 25 and all(cmd == 0 for cmd in command_history):
                        command_allowed = not command_allowed
                        if command_allowed == True:
                            print("ready")
                            threading.Thread(target=send_notes_to_server, args=(url, 0)).start()
                        if command_allowed == False:
                            print("end")
                            threading.Thread(target=send_notes_to_server, args=(url, 26)).start()
                        command_history.clear()
                    if command_allowed:
                        if len(command_history) == 25 and all(cmd == command_history[0] for cmd in command_history) and command!=last_command:
                            threading.Thread(target=send_notes_to_server, args=(url, command)).start()
                            last_command=command
                            print(command)
                            command_history.clear()
                    
            except Exception as e:
                noteManuale, noteTempo = "NoN", "NoN"


            mapped_noteManuale, mapped_noteTempo = map_values(noteManuale, noteTempo)

            display_note_box(image, mapped_noteManuale, mapped_noteTempo)

            # Display note
            # display_note_box(image, noteManuale, noteTempo)

            # Render detections
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
                )

            cv2.imshow('Mediapipe Feed', image)

            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()

# 6. Change Notes to Meanings for Player

In [7]:
def map_values(noteManuale, noteTempo):
    # Define the mappings for noteManuale and noteTempo
    manuale_mapping = {
        '3': 'ADD',
        '2': 'DEL',
        '5': 'MAX',
        '4': 'MIN'
    }

    tempo_mapping = {
        '3': 'PLUS',
        '2': 'MINUS',
        '5': 'DEF'
    }

    # Map the noteManuale and noteTempo values
    mapped_noteManuale = manuale_mapping.get(noteManuale, '???')
    mapped_noteTempo = tempo_mapping.get(noteTempo, '???')

    # Check for the ON/OFF condition
    return ('ON/OFF' if noteManuale == '0' and noteTempo == '0' else mapped_noteManuale,
            'ON/OFF' if noteManuale == '0' and noteTempo == '0' else mapped_noteTempo)

    return mapped_noteManuale, mapped_noteTempo

# 7. Send Notes to Server

In [8]:
def send_notes_to_server(url, command):
    try:
        requests.post(url, json={'number':command})
    except Exception as e:
        print(f"Error sending notes to server: {e}")

# 8. Display Note

In [9]:
# Helper function to display the note
def display_note_box(image, noteManuale, noteTempo):
    # Draw the box
    cv2.rectangle(image, (0, 0), (300, 73), (245, 117, 16), -1)

    # Display the text 'Note Manuale' and 'Note Tempo'
    cv2.putText(image, 'Manuale:', (10, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(image, noteManuale, (150, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    
    cv2.putText(image, 'Tempo:', (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(image, noteTempo, (150, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

# 9. Let's Start Play!

In [41]:
process_video()

ready
end
ready
end
ready
16
3
Error sending notes to server: HTTPConnectionPool(host='10.10.35.129', port=8080): Max retries exceeded with url: /producer (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x000001C0D88AB910>, 'Connection to 10.10.35.129 timed out. (connect timeout=None)'))
Error sending notes to server: HTTPConnectionPool(host='10.10.35.129', port=8080): Max retries exceeded with url: /producer (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x000001C0D8875CC0>, 'Connection to 10.10.35.129 timed out. (connect timeout=None)'))
Error sending notes to server: HTTPConnectionPool(host='10.10.35.129', port=8080): Max retries exceeded with url: /producer (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x000001C0D88B0F10>, 'Connection to 10.10.35.129 timed out. (connect timeout=None)'))
Error sending notes to server: HTTPConnectionPool(host='10.10.35.129', port=8080): Max retries exceeded with ur

KeyboardInterrupt: 

In [20]:
# Examples of usage:
print(combination_result(0, 0))  # Should return 0
print(combination_result(1, 0))  # Should return None
print(combination_result(1, 1))  # Should return 1
print(combination_result(1, 2))  # Should return 2
print(combination_result(1, 3))  # Should return 3
print(combination_result(1, 4))  # Should return 4
print(combination_result(1, 5))  # Should return 5
print(combination_result(2, 1))  # Should return 6
print(combination_result(2, 2))  # Should return 7
print(combination_result(2, 3))  # Should return 8
print(combination_result(2, 4))  # Should return 9
print(combination_result(2, 5))  # Should return 10
print(combination_result(2, 1))  # Should return 6
print(combination_result(0, 2))  # Should return None
print(combination_result(2, 0))  # Should return None
print(combination_result(3, 4))  # Should return 14

0
None
1
2
3
4
5
6
7
8
9
10
6
None
None
14
